# 02 · Labels, features and the final task list

**Project:** Acquisition shortcuts in public cancer imaging (authors: see README.md)
**Notebook written:** 28 Oct 2026 · **Needs:** the outputs of notebook 01 in `results/01_inventory/` · **Plan:** sections 6, 7, 7a

**What this notebook does, in plain words**
1. It downloads the official clinical label files for the six non-TCGA collections from the TCIA website and keeps them unchanged on Drive.
2. It turns every label source (GDC for TCGA/CPTAC, the six files, and "which collection" for the collection-defined tasks) into one table: task, patient, class.
3. It counts patients per class for every task and marks which tasks pass rule 7a.1 (at least 20 per class). This is the **final task list** for notebook 03.
4. It builds the **metadata features** per patient from the main series of notebook 01. Scanner/protocol and care-pathway features are kept apart (rule 7a.4).
5. It splits the free text of the scan descriptions into words and sorts every word into one of four groups (rule 7a.3):
   words that name the diagnosis (**blocked**), words that name a hospital (**blocked**),
   care-pathway words (kept apart), and scanner/protocol words (used).
6. It records which hospital (site) each patient comes from, where this is known, for the within-site analysis (RQ3).

**No images are downloaded and no model is fitted.** Outputs go to `results/02_tasks_features/`.

**How to run:** in Colab, *Runtime → Run all*. No GPU is needed. Sign in with **the Google account that holds the project Drive folder** when Drive asks. It takes a few minutes.

In [1]:
# 1. Setup: Drive and folders
import os, re, sys, json, time, gzip, hashlib, zipfile, datetime, platform, importlib, subprocess, collections
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT = Path('/content/drive/MyDrive/Acquisition-Shortcut-Audit')
else:  # local test runs only
    PROJECT = Path(os.environ['ASA_PROJECT_DIR'])
assert PROJECT.exists(), (f'Project folder not found: {PROJECT}. '
                          'Is Colab signed in to the Drive of the Google account that holds the project Drive folder?')

IN01 = PROJECT / 'results' / '01_inventory'
OUT = PROJECT / 'results' / '02_tasks_features'
LABELS = PROJECT / 'labels'
assert (IN01 / 'run_manifest.json').exists(), 'Run notebook 01 first: results/01_inventory/ is missing.'
for p in (OUT, LABELS):
    p.mkdir(parents=True, exist_ok=True)

RUN_STARTED = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
print('Project folder:', PROJECT)
print('Outputs to    :', OUT)
print('Run started   :', RUN_STARTED, '(UTC)')

Mounted at /content/drive
Project folder: /content/drive/MyDrive/Acquisition-Shortcut-Audit
Outputs to    : /content/drive/MyDrive/Acquisition-Shortcut-Audit/results/02_tasks_features
Run started   : 2026-10-28T15:29:58+00:00 (UTC)


In [2]:
# 2. Packages (Excel readers are installed only if missing)
for module, package in (('openpyxl', 'openpyxl'), ('xlrd', 'xlrd')):
    try:
        importlib.import_module(module)
    except ImportError:
        if not IN_COLAB:
            raise
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 70)

def show(df, n=30):
    """Display a table in Colab, or print it in a plain Python run."""
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n).to_string())

def md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

## Frozen settings
Fixed on 28 Oct 2026 after reviewing the outputs of notebook 01 and the six label files. Change them only with a dated entry in `00_LOG.md`.

In [3]:
# 3. Frozen settings
MIN_PER_CLASS = 20      # rule 7a.1, unchanged from notebook 01

# Rule 7a.5: no collection's dates were judged real in notebook 01 (dates_check.csv: every collection with a
# reference "looks shifted"; the others have constant or unverifiable years). Study year is therefore never a feature.
STUDY_YEAR_USED = False

# Official clinical files on the TCIA website (found on each collection's page, 28 Oct 2026)
TCIA_UPLOADS = 'https://www.cancerimagingarchive.net/wp-content/uploads/'
LABEL_FILES = {
    'Duke-Breast-Cancer-MRI': 'Clinical_and_Other_Features.xlsx',
    'UPENN-GBM': 'UPENN-GBM_clinical_info_v2.1.csv',
    'LIDC-IDRI': 'tcia-diagnosis-data-2012-04-20.xls',
    'PROSTATEx': 'ProstateX-TrainingLesionInformationv2.zip',
    'NSCLC Radiogenomics': 'NSCLCR01Radiogenomic_DATA_LABELS_2018-05-22_1500-shifted.csv',
    'HCC-TACE-Seg': 'HCC-TACE-Seg_clinical_data-V2.xlsx',
}

# Collection-defined tasks (plan 6b). Glioma grade is absent: TCGA-GBM/LGG have no CT/MR on TCIA or IDC (notebook 01).
COLLECTION_DEFINED_TASKS = {
    'lung histology (LUAD vs LUSC)': ['TCGA-LUAD', 'TCGA-LUSC'],
    'renal subtype (KIRC vs KIRP vs KICH)': ['TCGA-KIRC', 'TCGA-KIRP', 'TCGA-KICH'],
    'renal subtype (KIRC vs KIRP)': ['TCGA-KIRC', 'TCGA-KIRP'],
    'renal subtype (KIRC vs KICH)': ['TCGA-KIRC', 'TCGA-KICH'],
    'renal subtype (KIRP vs KICH)': ['TCGA-KIRP', 'TCGA-KICH'],
}
COLLECTION_DEFINED_MODALITY = 'CT'   # the only modality where these tasks have patients (task_eligibility.csv)

# Rule 7a.3: free-text words, reviewed by hand on 28 Oct 2026 from the 50 most frequent words per collection.
# Phrases removed before splitting into words:
BLOCKED_PHRASES = r'(?:high|low)[\s_-]*grade'
# Words that name a diagnosis, a class, a lesion or a molecular marker -> blocked
LABEL_WORDS = {
    'tumor', 'tumour', 'tumors', 'braintumor', 'cancer', 'ca', 'carcinoma', 'kanker', 'malignant', 'malignancy',
    'benign', 'neoplasm', 'mass', 'masses', 'lesion', 'lesions', 'nodule', 'nodules', 'mets', 'met', 'metastatic',
    'metastasis', 'metastases', 'grade', 'stage', 'gbm', 'lgg', 'glioma', 'glioblastoma', 'hcc', 'rcc', 'ccrcc',
    'kirc', 'kirp', 'kich', 'luad', 'lusc', 'adenocarcinoma', 'adeno', 'squamous', 'chromophobe', 'papillary',
    'her2', 'triple', 'tnbc', 'luminal', 'mgmt', 'idh', 'egfr', 'kras', 'onco', 'oncology', 'onc', 'tcga', 'cptac',
    'upenn', 'lidc', 'prostatex', 'endometrial', 'endometrio', 'lymphoma', 'melanoma', 'sarcoma', 'vhl'}
# Words that name a hospital or institution -> blocked (the site is used only for grouping, rule 7a.3)
SITE_WORDS = {'roswell', 'rpci', 'bjh', 'bwh', 'va', 'uob', 'fmc', 'osf', 'mskcc', 'mdacc', 'mayo', 'emory',
              'stanford', 'duke', 'penn', 'radboud', 'amc'}
# Words about why or when the patient was scanned -> kept apart as care pathway (rule 7a.4)
CARE_PATHWAY_WORDS = {'biopsy', 'bx', 'guided', 'screen', 'screening', 'nlst', 'acrin', 'ltrc', 'followup',
                      'follow', 'fu', 'restaging', 'staging', 'surveillance', 'preop', 'postop', 'stealth',
                      'outside', 'external', 'foreign', 'comparison', 'read', 'second', 'opinion', 'research',
                      'trial', 'hematuria', 'pain', 'jaundice', 'trauma', 'claustro', 'detectie', 'detection',
                      'kliniek', 'stone', 'workup', 'hx'}
# Series types that are annotations or analysis results, not acquisitions: left out of the care-pathway counts
NON_ACQUISITION_MODALITIES = {'SEG', 'RTSTRUCT', 'SR', 'RTDOSE', 'RTPLAN', 'PR', 'KO', 'REG', 'ANN', 'OT'}
# Words that mark processed (not acquired) images -> blocked; reported in token_review.csv
PROCESSING_WORDS = {'captk', 'processed'}

# CT reconstruction kernels and slice thickness parsed from the text (plan section 7)
# Siemens (B30f, I30f, Br34, Hr40), Toshiba (FC08). 'Dr' is left out on purpose: "DR40" means dose reduction.
KERNEL_RE = re.compile(r'\b((?:[bhiq]r|[bcdhiq]|fc)\d{2}[fsh]?)\b', re.I)
GE_KERNEL_WORDS = {'standard', 'std', 'stnd', 'soft', 'lung', 'bone', 'detail', 'edge'}
THICK_MM_RE = re.compile(r'(\d+(?:\.\d+)?)\s*mm\b', re.I)
# A number right before a kernel (e.g. "Thorax 1.0 B45f", "2.5 STANDARD") counts only if it is a usual slice
# thickness; GE protocol numbers such as "5.13 LUNG SCREEN" are not.
THICK_BEFORE_KERNEL_RE = re.compile(r'(\d+(?:\.\d+)?)\s+(?:(?:[bhiq]r|[bcdhiq]|fc)\d{2}[fsh]?|standard|std|stnd|soft|lung|bone|detail)\b', re.I)
USUAL_THICKNESSES = {0.5, 0.6, 0.625, 0.75, 0.8, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0, 3.75, 4.0, 5.0, 6.0, 7.0, 7.5, 8.0, 10.0}

# TCGA tissue-source-site codes name one institution for one cancer type, so the same hospital has different codes in
# different TCGA projects. GDC's code table maps each code to its institution (RQ3 needs the institution).
TSS_URL = 'https://gdc.cancer.gov/resources-tcga-users/tcga-code-tables/tissue-source-site-codes'
# Names are lower-cased and a leading biobank prefix such as "BLN - " or "ABS - " is removed. Then only spellings of
# the same institution that the table writes differently are merged; everything else stays as written.
INSTITUTION_PREFIX_RE = r'^[a-z]{2,4} - '
INSTITUTION_ALIASES = {'mskcc': 'memorial sloan kettering',
                       'memorial sloan kettering cancer center': 'memorial sloan kettering',
                       'mayo clinic - rochester': 'mayo clinic',
                       'mayo': 'mayo clinic',
                       'md anderson': 'md anderson cancer center',
                       'university of north carolina': 'unc',
                       'washington university - st. louis': 'washington university'}

## Inputs from notebook 01
Every file is checked against the checksums in notebook 01's run record, so we know nothing changed in between.

In [4]:
# 4. Load and verify notebook 01 outputs
man01 = json.loads((IN01 / 'run_manifest.json').read_text())
bad = [f['file'] for f in man01['outputs'] if md5(IN01 / f['file']) != f['md5']]
assert not bad, f'These notebook 01 files changed since that run: {bad}'
print('Notebook 01 outputs verified:', len(man01['outputs']), 'files, run of', man01['run_started_utc'])

series = pd.read_csv(IN01 / 'series_metadata.csv.gz', low_memory=False)
studies = pd.read_csv(IN01 / 'studies.csv.gz', low_memory=False)
main = pd.read_csv(IN01 / 'main_series.csv')
gdc = pd.read_csv(IN01 / 'gdc_labels.csv').drop_duplicates('PatientID')
primary_modality = man01['primary_modality']
print(f'{len(series):,} series, {len(studies):,} studies, {len(main):,} main series, {len(gdc):,} GDC patients')

Notebook 01 outputs verified: 18 files, run of 2026-10-28T12:20:42+00:00
63,468 series, 9,126 studies, 4,883 main series, 1,479 GDC patients


## Step A · Download the six official label files
Each file is downloaded once and kept unchanged under `labels/<collection>/`. A rerun reuses the saved file.

In [5]:
# 5. Step A: label files
SESSION = requests.Session()
SESSION.mount('https://', HTTPAdapter(max_retries=Retry(total=5, backoff_factor=1.5,
                                                        status_forcelist=(429, 500, 502, 503, 504))))
files = []
for col, name in LABEL_FILES.items():
    folder = LABELS / re.sub(r'\W+', '_', col)
    folder.mkdir(exist_ok=True)
    path = folder / name
    if not path.exists():
        r = SESSION.get(TCIA_UPLOADS + name, timeout=120)
        r.raise_for_status()
        tmp = path.with_suffix(path.suffix + '.part')
        tmp.write_bytes(r.content)
        tmp.replace(path)
        status = 'downloaded now'
    else:
        status = 'already on Drive'
    files.append({'collection': col, 'file': name, 'url': TCIA_UPLOADS + name, 'status': status,
                  'bytes': path.stat().st_size, 'md5': md5(path),
                  'checked_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')})
label_files = pd.DataFrame(files)
label_files.to_csv(OUT / 'label_files.csv', index=False)
show(label_files)

def label_path(col):
    return LABELS / re.sub(r'\W+', '_', col) / LABEL_FILES[col]

,collection,file,url,status,bytes,md5,checked_utc
0,Duke-Breast-Cancer-MRI,Clinical_and_Other_Features.xlsx,https://www.cancerimagingarchive.net/wp-content/uploads/Clinical_a...,downloaded now,595608,13d3a4833d945ff843453f8138517f65,2026-10-28T15:30:05+00:00
1,UPENN-GBM,UPENN-GBM_clinical_info_v2.1.csv,https://www.cancerimagingarchive.net/wp-content/uploads/UPENN-GBM_...,downloaded now,66491,160160309e355c5e815424e2e9e1c6da,2026-10-28T15:30:05+00:00
2,LIDC-IDRI,tcia-diagnosis-data-2012-04-20.xls,https://www.cancerimagingarchive.net/wp-content/uploads/tcia-diagn...,downloaded now,46080,00c00621419e16d96b92f19d63cd10f4,2026-10-28T15:30:05+00:00
3,PROSTATEx,ProstateX-TrainingLesionInformationv2.zip,https://www.cancerimagingarchive.net/wp-content/uploads/ProstateX-...,downloaded now,287527,0156733c14f2035cb5d9dbe0fa227829,2026-10-28T15:30:05+00:00
4,NSCLC Radiogenomics,NSCLCR01Radiogenomic_DATA_LABELS_2018-05-22_1500-shifted.csv,https://www.cancerimagingarchive.net/wp-content/uploads/NSCLCR01Ra...,downloaded now,66643,f0f65ae858c3677432f3fcb99553b78a,2026-10-28T15:30:05+00:00
5,HCC-TACE-Seg,HCC-TACE-Seg_clinical_data-V2.xlsx,https://www.cancerimagingarchive.net/wp-content/uploads/HCC-TACE-S...,downloaded now,128507,63af6363b9d0fe453c48e75402e9082c,2026-10-28T15:30:05+00:00


## Step B · Labels for every task
One row per task and patient. Only patients with a main series in the task's modality are kept.
Groupings that are not obvious are explained next to the code.

In [6]:
# 6. Step B: one long label table
rows = []
def add(task_id, task_type, label_source, collection, modality, labels):
    """labels: dict PatientID -> class (None = no usable label)."""
    have = set(main[(main.collection == collection) & (main.modality == modality)].PatientID)
    for pid, lab in labels.items():
        if lab is not None and pid in have:
            rows.append({'task_id': task_id, 'task_type': task_type, 'label_source': label_source,
                         'collection': collection, 'modality': modality, 'PatientID': pid, 'label': lab})

# 6a. TCGA and CPTAC: grade and stage from GDC (same groupings as notebook 01)
def grade_group(g):
    g = g.strip().lower() if isinstance(g, str) else None
    return {'g1': 'low (G1-G2)', 'g2': 'low (G1-G2)', 'low grade': 'low (G1-G2)',
            'g3': 'high (G3-G4)', 'g4': 'high (G3-G4)', 'high grade': 'high (G3-G4)'}.get(g)

def stage_group(s):
    m = re.match(r'\s*stage\s+([iv]+)', s.lower()) if isinstance(s, str) else None
    if not m:
        return None
    return {'i': 'early (I-II)', 'ii': 'early (I-II)', 'iii': 'late (III-IV)', 'iv': 'late (III-IV)'}.get(m.group(1))

gdc_map = gdc.set_index('PatientID')
for col in sorted(c for c in primary_modality if c.startswith(('TCGA-', 'CPTAC-'))):
    pids = main[main.collection == col].PatientID.unique()
    sub = gdc_map.reindex(pids)
    add(f'{col} | grade', 'within collection (6a)', 'GDC tumor_grade', col, primary_modality[col],
        {p: grade_group(g) for p, g in sub.tumor_grade.items()})
    add(f'{col} | stage', 'within collection (6a)', 'GDC ajcc_pathologic_stage', col, primary_modality[col],
        {p: stage_group(s) for p, s in sub.ajcc_pathologic_stage.items()})

# 6b. Collection-defined tasks: the class is the source collection
for task, cols in COLLECTION_DEFINED_TASKS.items():
    for col in cols:
        add(task, 'collection-defined (6b)', 'source collection', col, COLLECTION_DEFINED_MODALITY,
            {p: col for p in main[main.collection == col].PatientID.unique()})

# 6c. Duke-Breast-Cancer-MRI: row 1 holds the column names, row 2 the code legend
duke = pd.read_excel(label_path('Duke-Breast-Cancer-MRI'), sheet_name='Data', header=1).iloc[1:]
duke = duke.set_index('Patient ID')
mod = primary_modality['Duke-Breast-Cancer-MRI']
subtype = {0: 'luminal-like', 1: 'ER/PR+ HER2+', 2: 'HER2-enriched', 3: 'triple negative'}  # legend in the file
posneg = {0: 'negative', 1: 'positive'}
add('Duke-Breast-Cancer-MRI | molecular subtype', 'within collection (6a)', LABEL_FILES['Duke-Breast-Cancer-MRI'],
    'Duke-Breast-Cancer-MRI', mod, {p: subtype.get(v) for p, v in duke['Mol Subtype'].items()})
for marker in ('ER', 'PR', 'HER2'):
    add(f'Duke-Breast-Cancer-MRI | {marker} status', 'within collection (6a)', LABEL_FILES['Duke-Breast-Cancer-MRI'],
        'Duke-Breast-Cancer-MRI', mod, {p: posneg.get(v) for p, v in duke[marker].items()})

# 6d. UPENN-GBM: baseline pre-operative scans only (ID suffix _11)
up = pd.read_csv(label_path('UPENN-GBM'))
up = up[up.ID.str.endswith('_11')].assign(PatientID=lambda d: d.ID.str.replace(r'_11$', '', regex=True)).set_index('PatientID')
mod = primary_modality['UPENN-GBM']
add('UPENN-GBM | MGMT methylation', 'within collection (6a)', LABEL_FILES['UPENN-GBM'], 'UPENN-GBM', mod,
    {p: {'Methylated': 'methylated', 'Unmethylated': 'unmethylated'}.get(v) for p, v in up.MGMT.items()})
add('UPENN-GBM | IDH1 mutation', 'within collection (6a)', LABEL_FILES['UPENN-GBM'], 'UPENN-GBM', mod,
    {p: {'Mutated': 'mutated', 'Wildtype': 'wildtype'}.get(v) for p, v in up.IDH1.items()})

# 6e. LIDC-IDRI: patient-level confirmed diagnosis (0 unknown, 1 benign, 2 primary lung cancer, 3 metastatic)
lidc = pd.read_excel(label_path('LIDC-IDRI'))
dx_col = [c for c in lidc.columns if str(c).startswith('Diagnosis at the Patient Level')][0]
lidc = lidc.set_index('TCIA Patient ID')[dx_col]
mod = primary_modality['LIDC-IDRI']
add('LIDC-IDRI | benign vs malignant', 'within collection (6a)', LABEL_FILES['LIDC-IDRI'], 'LIDC-IDRI', mod,
    {p: {1: 'benign', 2: 'malignant', 3: 'malignant'}.get(v) for p, v in lidc.items()})
add('LIDC-IDRI | primary vs metastatic', 'within collection (6a)', LABEL_FILES['LIDC-IDRI'], 'LIDC-IDRI', mod,
    {p: {2: 'primary lung cancer', 3: 'metastatic'}.get(v) for p, v in lidc.items()})

# 6f. PROSTATEx: training set only (the public test set has no ClinSig label). A patient is "significant"
#     if any of their findings is clinically significant.
with zipfile.ZipFile(label_path('PROSTATEx')) as z:
    member = [n for n in z.namelist() if n.endswith('ProstateX-Findings-Train.csv')][0]
    px = pd.read_csv(z.open(member))
px_sig = px.assign(sig=px.ClinSig.astype(str).str.strip().str.lower().isin(['true', '1'])).groupby('ProxID').sig.any()
add('PROSTATEx | clinically significant', 'within collection (6a)', 'ProstateX-Findings-Train.csv', 'PROSTATEx',
    primary_modality['PROSTATEx'], {p: 'significant' if s else 'not significant' for p, s in px_sig.items()})

# 6g. NSCLC Radiogenomics
ns = pd.read_csv(label_path('NSCLC Radiogenomics'))
ns.columns = ns.columns.str.strip()
ns = ns.set_index('Case ID')
mod = primary_modality['NSCLC Radiogenomics']
src = LABEL_FILES['NSCLC Radiogenomics']
add('NSCLC Radiogenomics | histology', 'within collection (6a)', src, 'NSCLC Radiogenomics', mod,
    {p: {'Adenocarcinoma': 'adenocarcinoma', 'Squamous cell carcinoma': 'squamous'}.get(v) for p, v in ns.Histology.items()})
for gene in ('EGFR', 'KRAS'):
    add(f'NSCLC Radiogenomics | {gene} mutation', 'within collection (6a)', src, 'NSCLC Radiogenomics', mod,
        {p: {'Mutant': 'mutant', 'Wildtype': 'wildtype'}.get(v) for p, v in ns[f'{gene} mutation status'].items()})

# 6h. HCC-TACE-Seg: TNM stage, grouped like the TCGA stage tasks
hcc = pd.read_excel(label_path('HCC-TACE-Seg'), sheet_name='data table').set_index('TCIA_ID')
add('HCC-TACE-Seg | stage', 'within collection (6a)', LABEL_FILES['HCC-TACE-Seg'], 'HCC-TACE-Seg',
    primary_modality['HCC-TACE-Seg'], {p: stage_group(str(v).replace('-', ' ')) for p, v in hcc.TNM.items()})

labels_long = pd.DataFrame(rows)
labels_long.to_csv(OUT / 'labels_long.csv', index=False)
print(f'{len(labels_long):,} task-patient rows over {labels_long.task_id.nunique()} tasks')

8,467 task-patient rows over 52 tasks


## Step C · The final task list (rule 7a.1)

In [7]:
# 7. Step C: class counts and eligibility
tasks = []
for tid, g in labels_long.groupby('task_id', sort=False):
    vc = g.label.value_counts()
    tasks.append({'task_id': tid, 'task_type': g.task_type.iloc[0], 'label_source': g.label_source.iloc[0],
                  'collections': ', '.join(sorted(g.collection.unique())), 'modality': g.modality.iloc[0],
                  'n_patients': len(g), 'n_classes': len(vc),
                  'class_counts': '; '.join(f'{k}: {v}' for k, v in vc.items()),
                  'min_class_n': int(vc.min())})
tasks = pd.DataFrame(tasks)
tasks['eligible'] = (tasks.n_classes >= 2) & (tasks.min_class_n >= MIN_PER_CLASS)
tasks = tasks.sort_values(['eligible', 'task_type', 'task_id'], ascending=[False, True, True])
tasks.to_csv(OUT / 'tasks_final.csv', index=False)
print(f'Eligible tasks: {int(tasks.eligible.sum())} of {len(tasks)} '
      f'({int(tasks[tasks.eligible].task_type.str.startswith("within").sum())} within collection, '
      f'{int(tasks[tasks.eligible].task_type.str.startswith("collection").sum())} collection-defined)')
show(tasks[['task_id', 'modality', 'n_patients', 'class_counts', 'eligible']], 60)

Eligible tasks: 26 of 52 (24 within collection, 2 collection-defined)


,task_id,modality,n_patients,class_counts,eligible
34,lung histology (LUAD vs LUSC),CT,104,TCGA-LUAD: 67; TCGA-LUSC: 37,True
36,renal subtype (KIRC vs KIRP),CT,259,TCGA-KIRC: 236; TCGA-KIRP: 23,True
0,CPTAC-CCRCC | grade,CT,63,low (G1-G2): 43; high (G3-G4): 20,True
1,CPTAC-CCRCC | stage,CT,64,early (I-II): 41; late (III-IV): 23,True
7,CPTAC-PDA | grade,CT,71,low (G1-G2): 51; high (G3-G4): 20,True
8,CPTAC-PDA | stage,CT,68,early (I-II): 47; late (III-IV): 21,True
11,CPTAC-UCEC | grade,CT,69,low (G1-G2): 49; high (G3-G4): 20,True
40,Duke-Breast-Cancer-MRI | ER status,MR,922,positive: 686; negative: 236,True
42,Duke-Breast-Cancer-MRI | HER2 status,MR,922,negative: 759; positive: 163,True
41,Duke-Breast-Cancer-MRI | PR status,MR,922,positive: 598; negative: 324,True


## Step D · Metadata features per patient
From each patient's main series (notebook 01, rule 7a.7). **Scanner/protocol** features describe how the main series was
acquired. **Care-pathway** features describe how much imaging the patient had and when (rule 7a.4); they are kept apart
because they can differ by disease for legitimate clinical reasons.

In [8]:
# 8. Step D: scanner/protocol and care-pathway features
ms = main.merge(series[['SeriesInstanceUID', 'StudyDesc', 'BodyPartExamined', 'FileSize']], on='SeriesInstanceUID', how='left')
text = ms[['SeriesDescription', 'ProtocolName']].fillna('').agg(' | '.join, axis=1)

def parse_thickness(t):
    m = THICK_MM_RE.search(t)
    if m:
        v = float(m.group(1))
        return v if 0.4 <= v <= 10 else np.nan     # outside this range it is not a slice thickness
    m = THICK_BEFORE_KERNEL_RE.search(t)
    v = float(m.group(1)) if m else np.nan
    return v if v in USUAL_THICKNESSES else np.nan

def parse_kernel(t, modality, vendor):
    if modality != 'CT':
        return np.nan
    m = KERNEL_RE.search(t)
    if m:
        return m.group(1).upper()
    if vendor == 'GE':
        words = set(re.split(r'[^a-z]+', t.lower())) & GE_KERNEL_WORDS
        return '/'.join(sorted(words)).upper() if words else np.nan
    return np.nan

feat = pd.DataFrame({
    'collection': ms.collection, 'PatientID': ms.PatientID, 'modality': ms.modality,
    'is_primary_modality': ms.is_primary_modality,
    # scanner / protocol (rule 7a.4, primary audit)
    'vendor': ms.Manufacturer.str.upper().str.extract(r'^\s*([A-Z]+)', expand=False),
    'manufacturer': ms.Manufacturer.str.strip(),
    'model': ms.ManufacturerModelName.str.strip().str.upper(),
    'software_version': ms.SoftwareVersions.astype('string').str.strip(),
    'image_count': ms.ImageCount,
    'body_part': ms.BodyPartExamined,
    'slice_thickness_mm': text.map(parse_thickness),
})
feat['kernel'] = [parse_kernel(t, m, v) for t, m, v in zip(text, feat.modality, feat.vendor)]
feat['file_size_mb'] = ms.FileSize / 1e6          # sensitivity analysis only (rule 7a.3)

# care pathway (rule 7a.4, reported separately); acquisitions only
third_party = series.ThirdPartyAnalysis.astype(str).str.strip().str.lower().eq('yes') if 'ThirdPartyAnalysis' in series else False
acq = series[~series.Modality.isin(NON_ACQUISITION_MODALITIES) & ~third_party]
cp = (acq.groupby(['Collection', 'PatientID'])
         .agg(cp_n_studies=('StudyInstanceUID', 'nunique'), cp_n_series=('SeriesInstanceUID', 'size'),
              cp_n_series_CT=('Modality', lambda m: int((m == 'CT').sum())),
              cp_n_series_MR=('Modality', lambda m: int((m == 'MR').sum())),
              cp_has_PT=('Modality', lambda m: bool((m == 'PT').any())),
              cp_n_modalities=('Modality', 'nunique'))
         .reset_index().rename(columns={'Collection': 'collection'}))
feat = feat.merge(cp, on=['collection', 'PatientID'], how='left')
feat['cp_n_studies_this_modality'] = ms.n_studies_this_modality.values
feat['cp_study_rank_used'] = ms.study_rank_used.values
# days from diagnosis to the main study, where TCIA provides it (not a calendar date, so rule 7a.5 does not apply)
if {'LongitudinalTemporalEventType', 'LongitudinalTemporalOffsetFromEvent'} <= set(studies.columns):
    off = studies[studies.LongitudinalTemporalEventType.astype(str).str.upper() == 'DIAGNOSIS']
    off = off.drop_duplicates('StudyInstanceUID').set_index('StudyInstanceUID').LongitudinalTemporalOffsetFromEvent
    feat['cp_days_from_diagnosis'] = pd.to_numeric(ms.StudyInstanceUID.map(off), errors='coerce').values
feat.to_csv(OUT / 'patient_features.csv', index=False)

fcov = (feat[feat.is_primary_modality].groupby('collection')
        [['vendor', 'model', 'software_version', 'image_count', 'slice_thickness_mm', 'kernel']]
        .agg(lambda s: round(float(s.notna().mean()), 3)).add_prefix('filled_').reset_index())
fcov.to_csv(OUT / 'feature_coverage.csv', index=False)
print(f'Features for {len(feat):,} patient-modality rows. Coverage in the primary modality:')
show(fcov, 40)

Features for 4,883 patient-modality rows. Coverage in the primary modality:


,collection,filled_vendor,filled_model,filled_software_version,filled_image_count,filled_slice_thickness_mm,filled_kernel
0,CPTAC-AML,1.000,1.000,1.000,1.0,0.600,0.000
1,CPTAC-CCRCC,1.000,1.000,0.984,1.0,0.359,0.375
2,CPTAC-CM,0.647,1.000,1.000,1.0,0.647,0.529
3,CPTAC-LSCC,0.611,1.000,1.000,1.0,0.222,0.278
4,CPTAC-LUAD,0.946,1.000,1.000,1.0,0.378,0.514
5,CPTAC-PDA,0.825,1.000,0.907,1.0,0.247,0.227
6,CPTAC-SAR,0.889,1.000,1.000,1.0,0.222,0.278
7,CPTAC-STAD,1.000,1.000,1.000,1.0,0.105,0.053
8,CPTAC-UCEC,0.693,1.000,1.000,1.0,0.800,0.773
9,Duke-Breast-Cancer-MRI,1.000,1.000,1.000,1.0,0.000,0.000


## Step E · Words from the scan descriptions (rule 7a.3)
The text of the main series (SeriesDescription, ProtocolName, StudyDesc) is split into words. Each word is sorted into:
**label** (names a diagnosis or class: blocked), **site** (names a hospital: blocked), **processing** (marks processed
images: blocked), **care pathway** (kept apart) or **protocol** (used). `token_review.csv` keeps the 50 most frequent words
per collection with their group, as the record of the review by hand.

In [9]:
# 9. Step E: tokens
def category(tok):
    if tok in LABEL_WORDS:
        return 'label (blocked)'
    if tok in SITE_WORDS:
        return 'site (blocked)'
    if tok in PROCESSING_WORDS:
        return 'processing (blocked)'
    if tok in CARE_PATHWAY_WORDS:
        return 'care pathway'
    return 'protocol'

def tokenize(t):
    if not isinstance(t, str):
        return set()
    t = re.sub(BLOCKED_PHRASES, ' grade ', t.lower())
    return {w.strip('.') for w in re.split(r'[^a-z0-9.]+', t) if w.strip('.')}

tok_rows = []
for r in ms.itertuples():
    found = {}
    for field in ('SeriesDescription', 'ProtocolName', 'StudyDesc'):
        for w in tokenize(getattr(r, field)):
            found.setdefault(w, field)
    for w, field in found.items():
        tok_rows.append({'collection': r.collection, 'PatientID': r.PatientID, 'modality': r.modality,
                         'is_primary_modality': r.is_primary_modality, 'token': w, 'first_field': field,
                         'category': category(w)})
tokens_all = pd.DataFrame(tok_rows)
kept = tokens_all[~tokens_all.category.str.endswith('(blocked)')]
kept.drop(columns='is_primary_modality').to_csv(OUT / 'patient_tokens.csv.gz', index=False)

prim = tokens_all[tokens_all.is_primary_modality]
counts = prim.groupby(['collection', 'token', 'category']).PatientID.nunique().rename('n_patients').reset_index()
review = (counts.sort_values(['collection', 'n_patients', 'token'], ascending=[True, False, True])
                .groupby('collection').head(50))
review.to_csv(OUT / 'token_review.csv', index=False)
blocked = counts[counts.category.str.endswith('(blocked)')].sort_values(['collection', 'n_patients'], ascending=[True, False])
blocked.to_csv(OUT / 'tokens_blocked.csv', index=False)
print(f'{tokens_all.token.nunique():,} distinct words; blocked word-patient pairs: '
      f'{int((tokens_all.category.str.endswith("(blocked)")).sum()):,}')
print(tokens_all.drop_duplicates(['PatientID', 'modality', 'token']).category.value_counts().to_string())

1,374 distinct words; blocked word-patient pairs: 1,884
category
protocol                33978
care pathway             1038
processing (blocked)     1018
label (blocked)           809
site (blocked)             57


In [10]:
# 10. Possible hospital words not on the list: words used by 3+ patients who all come from one TCGA site,
#     in a collection with at least two sites. These are flagged for review before notebook 03, not blocked here.
t_prim = prim[prim.collection.str.startswith('TCGA-') & (prim.category == 'protocol')].copy()
t_prim['site'] = t_prim.PatientID.str.split('-').str[1]
n_sites = (main[main.collection.str.startswith('TCGA-')].assign(site=lambda d: d.PatientID.str.split('-').str[1])
           .groupby('collection').site.nunique())
cand = (t_prim.groupby(['collection', 'token'])
              .agg(n_patients=('PatientID', 'nunique'), n_sites=('site', 'nunique'), site=('site', 'first'))
              .reset_index())
cand = cand[(cand.n_patients >= 3) & (cand.n_sites == 1) & cand.collection.map(n_sites).ge(2)]
cand.sort_values(['collection', 'n_patients'], ascending=[True, False]).to_csv(OUT / 'site_token_candidates.csv', index=False)
print('Words confined to a single TCGA site (for review):', len(cand))

Words confined to a single TCGA site (for review): 221


## Step F · Sites (for the within-site analysis, RQ3)
Known for TCGA and NSCLC Radiogenomics; the other non-TCGA collections come from a single site or do not say.
- **TCGA:** the tissue-source-site code (2nd part of the barcode) is mapped to its **institution** with GDC's code table.
  A code names one institution *for one cancer type*, so only the institution can show the same hospital across
  TCGA projects (needed for the collection-defined tasks). Both levels are kept.
- **NSCLC Radiogenomics:** the "Patient affiliation" column of the clinical file (Stanford or VA).

In [11]:
# 11. Step F: GDC's table of tissue-source-site codes (saved once to labels/, then reused)
import io
tss_path = LABELS / 'GDC_tissue_source_site_codes.html'
if not tss_path.exists():
    r = SESSION.get(TSS_URL, timeout=120)
    r.raise_for_status()
    tss_path.write_text(r.text, encoding='utf-8')
tss_tables = [t for t in pd.read_html(io.StringIO(tss_path.read_text(encoding='utf-8')))
              if 'TSS Code' in map(str, t.columns)]
assert tss_tables, 'The GDC page no longer has a "TSS Code" table; check it by hand.'
tss = tss_tables[0].rename(columns={'TSS Code': 'tss', 'Source Site': 'institution_raw', 'Study Name': 'study'})
tss['tss'] = tss.tss.astype(str).str.strip().str.zfill(2)
tss['institution'] = (tss.institution_raw.astype(str).str.strip().str.lower()
                      .str.replace(INSTITUTION_PREFIX_RE, '', regex=True)
                      .map(lambda s: INSTITUTION_ALIASES.get(s, s)))
tss = tss.drop_duplicates('tss').set_index('tss')
files.append({'collection': 'TCGA (all)', 'file': tss_path.name, 'url': TSS_URL, 'bytes': tss_path.stat().st_size,
              'md5': md5(tss_path), 'status': 'saved web page'})
pd.DataFrame(files).to_csv(OUT / 'label_files.csv', index=False)
print(f'GDC code table: {len(tss)} site codes, {tss.institution.nunique()} institutions')

GDC code table: 830 site codes, 202 institutions


In [12]:
# 12. Step F: sites per patient and per task
sites = []
for pid, col in main[main.collection.str.startswith('TCGA-')][['PatientID', 'collection']].drop_duplicates().itertuples(index=False):
    code = pid.split('-')[1]
    sites.append({'collection': col, 'PatientID': pid, 'site_source': 'TCGA tissue source site', 'tss': code,
                  'institution_raw': tss.institution_raw.get(code),
                  'institution': tss.institution.get(code, f'unknown code {code}')})
if 'Patient affiliation' in ns.columns:
    for pid, aff in ns['Patient affiliation'].items():
        sites.append({'collection': 'NSCLC Radiogenomics', 'PatientID': pid,
                      'site_source': 'clinical file: Patient affiliation', 'tss': None,
                      'institution_raw': aff, 'institution': aff})
sites = pd.DataFrame(sites)
sites.to_csv(OUT / 'patient_sites.csv', index=False)
print('TCGA patients whose site code is missing from the GDC table:',
      int(sites.institution.astype(str).str.startswith('unknown code').sum()))

ss = []
for tid, g in labels_long[labels_long.task_id.isin(tasks[tasks.eligible].task_id)].groupby('task_id'):
    g = g.merge(sites[['PatientID', 'tss', 'institution']], on='PatientID', how='left')
    for level in ('tss', 'institution'):
        if g[level].isna().all():
            continue
        per = g.groupby(level).label.value_counts().unstack(fill_value=0)
        both = per[(per >= 5).sum(axis=1) >= 2]    # sites with at least 5 patients in at least two classes
        ss.append({'task_id': tid, 'site_level': level, 'n_sites': len(per),
                   'n_sites_5_per_class': len(both), 'patients_in_those_sites': int(both.values.sum()),
                   'largest_sites': '; '.join(f'{s}: {int(n)}' for s, n in
                                              per.sum(axis=1).sort_values(ascending=False).head(4).items())})
site_summary = pd.DataFrame(ss)
site_summary.to_csv(OUT / 'site_summary.csv', index=False)
print('Eligible tasks with no site information:',
      sorted(set(tasks[tasks.eligible].task_id) - set(site_summary.task_id)))
show(site_summary, 40)

TCGA patients whose site code is missing from the GDC table: 0
Eligible tasks with no site information: ['CPTAC-CCRCC | grade', 'CPTAC-CCRCC | stage', 'CPTAC-PDA | grade', 'CPTAC-PDA | stage', 'CPTAC-UCEC | grade', 'Duke-Breast-Cancer-MRI | ER status', 'Duke-Breast-Cancer-MRI | HER2 status', 'Duke-Breast-Cancer-MRI | PR status', 'Duke-Breast-Cancer-MRI | molecular subtype', 'HCC-TACE-Seg | stage', 'LIDC-IDRI | benign vs malignant', 'LIDC-IDRI | primary vs metastatic', 'PROSTATEx | clinically significant', 'UPENN-GBM | MGMT methylation']


,task_id,site_level,n_sites,n_sites_5_per_class,patients_in_those_sites,largest_sites
0,NSCLC Radiogenomics | EGFR mutation,institution,2,2,166,Stanford: 91; VA: 75
1,NSCLC Radiogenomics | KRAS mutation,institution,2,2,165,Stanford: 91; VA: 74
2,NSCLC Radiogenomics | histology,institution,2,2,199,Stanford: 109; VA: 90
3,TCGA-BLCA | stage,tss,8,4,72,FD: 21; ZF: 21; DK: 19; G2: 14
4,TCGA-BLCA | stage,institution,8,4,72,university of sheffield: 21; university of chicago: 21; memorial s...
5,TCGA-BRCA | stage,tss,7,2,83,BH: 46; E2: 37; OL: 17; AO: 13
6,TCGA-BRCA | stage,institution,7,2,83,university of pittsburgh: 46; roswell park: 37; university of chic...
7,TCGA-KIRC | grade,tss,8,6,224,BP: 88; CJ: 51; B0: 30; B8: 25
8,TCGA-KIRC | grade,institution,8,6,224,memorial sloan kettering: 88; md anderson cancer center: 51; unive...
9,TCGA-KIRC | stage,tss,8,5,209,BP: 88; CJ: 51; B0: 30; B8: 25


## Step G · Run record and summary

In [13]:
# 12. Step G: manifest (what was run, and a checksum for every output file)
outputs = []
for p in sorted(OUT.iterdir()):
    if p.name == 'run_manifest.json':
        continue
    n_rows = len(pd.read_csv(p, low_memory=False)) if (p.suffix == '.csv' or p.name.endswith('.csv.gz')) else None
    outputs.append({'file': p.name, 'bytes': p.stat().st_size, 'rows': n_rows, 'md5': md5(p)})
manifest = {
    'notebook': '02_labels_features_tasks.ipynb', 'run_started_utc': RUN_STARTED,
    'run_finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'in_colab': IN_COLAB, 'input_notebook01_run': man01['run_started_utc'],
    'label_files': files,
    'settings': {'MIN_PER_CLASS': MIN_PER_CLASS, 'STUDY_YEAR_USED': STUDY_YEAR_USED,
                 'COLLECTION_DEFINED_TASKS': COLLECTION_DEFINED_TASKS,
                 'COLLECTION_DEFINED_MODALITY': COLLECTION_DEFINED_MODALITY,
                 'BLOCKED_PHRASES': BLOCKED_PHRASES, 'LABEL_WORDS': sorted(LABEL_WORDS),
                 'SITE_WORDS': sorted(SITE_WORDS), 'CARE_PATHWAY_WORDS': sorted(CARE_PATHWAY_WORDS),
                 'PROCESSING_WORDS': sorted(PROCESSING_WORDS),
                 'NON_ACQUISITION_MODALITIES': sorted(NON_ACQUISITION_MODALITIES), 'KERNEL_RE': KERNEL_RE.pattern,
                 'GE_KERNEL_WORDS': sorted(GE_KERNEL_WORDS), 'THICK_MM_RE': THICK_MM_RE.pattern,
                 'THICK_BEFORE_KERNEL_RE': THICK_BEFORE_KERNEL_RE.pattern,
                 'USUAL_THICKNESSES': sorted(USUAL_THICKNESSES), 'TSS_URL': TSS_URL,
                 'INSTITUTION_PREFIX_RE': INSTITUTION_PREFIX_RE, 'INSTITUTION_ALIASES': INSTITUTION_ALIASES},
    'versions': {'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__},
    'outputs': outputs,
}
(OUT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
show(pd.DataFrame(outputs), 40)

,file,bytes,rows,md5
0,feature_coverage.csv,1405,33,3ccd8cb74b35129457759a513129893b
1,label_files.csv,1619,7,758b0fbef645d4dad52b9d588b282677
2,labels_long.csv,1084633,8467,ef27be958776129af5060648984f7a5a
3,patient_features.csv,592135,4883,dc1c6ce8aff7a9b4495a787d39a5ec9e
4,patient_sites.csv,117088,1354,d21e52453f4fcfff0801e943f9ff387a
5,patient_tokens.csv.gz,129486,35016,ad5c3d47e8acb343cebbb9f30c941bf2
6,site_summary.csv,2027,21,369a89702f83a4102fd2047878e6ee3b
7,site_token_candidates.csv,4913,221,f426c3e61901f360459c37c7483d0c22
8,tasks_final.csv,7184,52,c7c0f74e4fa9cdda28dde0261843cea4
9,token_review.csv,40648,1471,1699355cd2659ddd85e8f36d55369caf


In [14]:
# 13. Plain-English summary, read back from the saved files
t = pd.read_csv(OUT / 'tasks_final.csv')
e = t[t.eligible]
print(f"- Tasks defined: {len(t)}; eligible under rule 7a.1 (>= {MIN_PER_CLASS} patients per class): {len(e)}.")
for tt, g in e.groupby('task_type'):
    print(f"  - {tt}: {len(g)} tasks, {int(g.n_patients.sum()):,} task-patient pairs")
print(f"- Label files: {', '.join(pd.read_csv(OUT / 'label_files.csv').status.value_counts().index)}.")
print(f"- Study year is not used anywhere (rule 7a.5).")
print('\nNext: review token_review.csv and site_token_candidates.csv,')
print('then writes notebook 03 (the metadata audit).')

- Tasks defined: 52; eligible under rule 7a.1 (>= 20 patients per class): 26.
  - collection-defined (6b): 2 tasks, 363 task-patient pairs
  - within collection (6a): 24 tasks, 6,276 task-patient pairs
- Label files: downloaded now, saved web page.
- Study year is not used anywhere (rule 7a.5).

Next: review token_review.csv and site_token_candidates.csv,
then writes notebook 03 (the metadata audit).
